In [1]:
import math

import rclpy
from rclpy.node import Node
from geometry_msgs.msg import PoseStamped, PoseWithCovariance
from transformations import euler_from_quaternion

from nav_msgs.msg import Odometry

from tf2_ros.transform_listener import TransformListener
from tf2_ros.buffer import Buffer

from threading import Lock

In [2]:
class HumanFollower(Node):
    def __init__(self):
        super().__init__('human_follower')

        self.human_sub = self.create_subscription(
            PoseStamped,
            'human_pose',
            self.human_callback,
            10
        )

        # self.robot_sub = self.create_subscription(
        #     Odometry,
        #     'odom',
        #     self.robot_callback,
        #     qos_profile=rclpy.qos.QoSProfile(
        #         history=rclpy.qos.QoSHistoryPolicy.KEEP_LAST,
        #         depth=1,
        #         durability=rclpy.qos.QoSDurabilityPolicy.VOLATILE,
        #         reliability=rclpy.qos.QoSReliabilityPolicy.BEST_EFFORT
        #     ),
        # )

        self.tf_buffer = Buffer()
        self.tf_listener = TransformListener(self.tf_buffer, self)

        self.goal_pub = self.create_publisher(
            PoseStamped,
            'follow_goal',
            10
        )

        self.human_pose = None
        self.robot_pose = None

        # Desired distance behind the human (meters)
        self.follow_distance = 1.0

        self._timer = self.create_timer(1.0, self.get_base_link_tf)
        self.lock = Lock()

    def get_base_link_tf(self):
        with self.lock:
            try:
                transform = self.tf_buffer.lookup_transform(
                        "map",  # target frame
                        "base_link",        # source frame
                        rclpy.time.Time(),
                        timeout=rclpy.duration.Duration(seconds=0.1)
                    )
    
                self.robot_pose = PoseStamped()
                self.robot_pose.pose.position.x = transform.transform.translation.x
                self.robot_pose.pose.position.y = transform.transform.translation.y
            except:
                pass
    
    # def robot_callback(self, msg: Odometry):
    #     self.robot_pose = msg.pose

    def human_callback(self, msg: PoseStamped):
        self.human_pose = msg
        self.compute_goal()

    def compute_goal(self):
        if self.human_pose is None or self.robot_pose is None:
            return

        # --- Human position ---
        hx = self.human_pose.pose.position.x
        hy = self.human_pose.pose.position.y

        with self.lock:
            # --- Robot position ---
            rx = self.robot_pose.pose.position.x
            ry = self.robot_pose.pose.position.y

        # --- Human yaw ---
        q = self.human_pose.pose.orientation
        _, _, human_yaw = euler_from_quaternion(
            [q.x, q.y, q.z, q.w]
        )

        dx = hx - rx
        dy = hy - ry
        distance = math.sqrt(dx*dx + dy*dy)

        if distance < 1e-6:
            # Robot is at human position, just stay
            gx, gy = rx, ry
            self.get_logger().info("Distance is too short.")
        else:
            # Scale vector to maintain desired distance
            scale = max(distance - self.follow_distance, 0.0) / distance
            gx = rx + dx * scale
            gy = ry + dy * scale

            self.get_logger().info("With robot X: {}; Y: {}.".format(
                rx, ry
            ))

        goal = PoseStamped()
        goal.header = self.human_pose.header

        goal.pose.position.x = gx
        goal.pose.position.y = gy
        goal.pose.position.z = 0.0

        # Optional: face the human
        # goal.pose.orientation = self.human_pose.pose.orientation

        yaw = math.atan2(gy - ry, gx - rx)
        goal.pose.orientation.z = math.sin(yaw / 2)
        goal.pose.orientation.w = math.cos(yaw / 2)

        self.goal_pub.publish(goal)

In [ ]:
rclpy.init()
node = HumanFollower()
rclpy.spin(node)
node.destroy_node()
rclpy.shutdown()

[INFO] [1771938040.313276298] [human_follower]: With robot X: -1.1238299341163556; Y: 0.3613241410168806.
[INFO] [1771938040.343259510] [human_follower]: With robot X: -1.1238299341163556; Y: 0.3613241410168806.
[INFO] [1771938040.443741472] [human_follower]: With robot X: -1.1238299341163556; Y: 0.3613241410168806.
[INFO] [1771938040.760452911] [human_follower]: With robot X: -1.1238299341163556; Y: 0.3613241410168806.
[INFO] [1771938040.846606596] [human_follower]: With robot X: -1.1238299341163556; Y: 0.3613241410168806.
[INFO] [1771938040.948375987] [human_follower]: With robot X: -1.1238299341163556; Y: 0.3613241410168806.
[INFO] [1771938041.248823670] [human_follower]: With robot X: -1.1238299341163556; Y: 0.3613241410168806.
[INFO] [1771938041.342730791] [human_follower]: With robot X: -1.1238299341163551; Y: 0.3613241410168806.
[INFO] [1771938041.439957845] [human_follower]: With robot X: -1.1238299341163551; Y: 0.3613241410168806.
[INFO] [1771938041.750858268] [human_follower]